# 2:4 Block Sparsity Sanity Check

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.utils.parametrize as parametrize

from torch_weighttracker import WeightTracker
from torch_weighttracker.calculations import CalcType
from torch_weighttracker.trackers import TrackerType

torch.set_grad_enabled(False)


def linear_with_weight(weight):
    model = nn.Sequential(nn.Linear(weight.shape[1], weight.shape[0], bias=False))
    with torch.no_grad():
        model[0].weight.copy_(weight)
    return model


def block_2_4_counts(model):
    return WeightTracker(model).get_calculation(CalcType.BLOCK_2_4_SPARSITY)()


def track_2_4(model, **kwargs):
    return WeightTracker(model).create_tracker(
        TrackerType.NVIDIA_2_4_SPARSITY,
        **kwargs,
    ).track()


def scalar(value):
    if isinstance(value, torch.Tensor):
        return float(value.detach().cpu())
    return value


def selected_scalars(metrics, keys):
    return {key: scalar(metrics[key]) for key in keys}


def count_row(result, index=0):
    strict, eligible, total, tails = result[index].detach().cpu().tolist()
    return {
        "strict_blocks": strict,
        "nvidia_eligible_blocks": eligible,
        "total_blocks": total,
        "tail_elements": tails,
    }


class FakeQuantZeroSmall(nn.Module):
    def forward(self, weight):
        return torch.where(weight.abs() < 0.5, torch.zeros_like(weight), weight)


assert CalcType.BLOCK_2_4_SPARSITY.value == "2_4_block_sparsity"
assert TrackerType.NVIDIA_2_4_SPARSITY.value == "nvidia_2_4_sparsity"

{
    "calculation": CalcType.BLOCK_2_4_SPARSITY.value,
    "tracker": TrackerType.NVIDIA_2_4_SPARSITY.value,
}

## Linear Strict, Eligible, And Invalid Blocks

In [ ]:
linear_model = linear_with_weight(
    torch.tensor(
        [
            [
                0.0, 0.0, 1.0, 2.0,  # strict: exactly two zeros
                3.0, 0.0, 4.0, 0.0,  # strict: zero positions may vary
                0.0, 0.0, 0.0, 1.0,  # eligible: at least two zeros
                0.0, 1.0, 2.0, 3.0,  # invalid: fewer than two zeros
            ]
        ]
    )
)

linear_counts = block_2_4_counts(linear_model)
linear_metrics = track_2_4(linear_model)

torch.testing.assert_close(linear_counts, torch.tensor([[2.0, 3.0, 4.0, 0.0]]))
torch.testing.assert_close(
    linear_metrics["nvidia_2_4_sparsity/strict_block_fraction"],
    torch.tensor(0.5),
)
torch.testing.assert_close(
    linear_metrics["nvidia_2_4_sparsity/nvidia_eligible_block_fraction"],
    torch.tensor(0.75),
)
torch.testing.assert_close(
    linear_metrics["nvidia_2_4_sparsity/strict_layers"],
    torch.tensor(0.0),
)
torch.testing.assert_close(
    linear_metrics["nvidia_2_4_sparsity/nvidia_eligible_layers"],
    torch.tensor(0.0),
)

{
    "counts": count_row(linear_counts),
    "metrics": selected_scalars(
        linear_metrics,
        [
            "nvidia_2_4_sparsity/strict_block_fraction",
            "nvidia_2_4_sparsity/nvidia_eligible_block_fraction",
            "nvidia_2_4_sparsity/strict_layers",
            "nvidia_2_4_sparsity/nvidia_eligible_layers",
        ],
    ),
}

## Tail Elements

In [ ]:
tail_model = linear_with_weight(torch.tensor([[0.0, 0.0, 1.0, 2.0, 0.0, 0.0]]))

tail_counts = block_2_4_counts(tail_model)
tail_metrics = track_2_4(tail_model)

torch.testing.assert_close(tail_counts, torch.tensor([[1.0, 1.0, 1.0, 2.0]]))
torch.testing.assert_close(
    tail_metrics["nvidia_2_4_sparsity/strict_block_fraction"],
    torch.tensor(1.0),
)
torch.testing.assert_close(
    tail_metrics["nvidia_2_4_sparsity/nvidia_eligible_block_fraction"],
    torch.tensor(1.0),
)
torch.testing.assert_close(
    tail_metrics["nvidia_2_4_sparsity/strict_layers"],
    torch.tensor(0.0),
)
torch.testing.assert_close(
    tail_metrics["nvidia_2_4_sparsity/nvidia_eligible_layers"],
    torch.tensor(0.0),
)
torch.testing.assert_close(
    tail_metrics["nvidia_2_4_sparsity/tail_elements"],
    torch.tensor(2.0),
)

{
    "counts": count_row(tail_counts),
    "metrics": selected_scalars(
        tail_metrics,
        [
            "nvidia_2_4_sparsity/strict_block_fraction",
            "nvidia_2_4_sparsity/nvidia_eligible_block_fraction",
            "nvidia_2_4_sparsity/strict_layers",
            "nvidia_2_4_sparsity/nvidia_eligible_layers",
            "nvidia_2_4_sparsity/tail_elements",
        ],
    ),
}

## Conv2d Input-Channel Blocks

In [ ]:
conv_model = nn.Sequential(nn.Conv2d(4, 2, kernel_size=2, bias=False))
with torch.no_grad():
    conv_model[0].weight.fill_(1.0)
    # TensorRT checks groups of four C-axis values for each [K, R, S] site.
    conv_model[0].weight[0, 0:2, :, :] = 0.0  # four strict C-axis blocks
    conv_model[0].weight[1, :, :, :] = 0.0  # four eligible-only C-axis blocks

conv_counts = block_2_4_counts(conv_model)
conv_metrics = track_2_4(conv_model)

torch.testing.assert_close(conv_counts, torch.tensor([[4.0, 8.0, 8.0, 0.0]]))
torch.testing.assert_close(
    conv_metrics["nvidia_2_4_sparsity/strict_block_fraction"],
    torch.tensor(0.5),
)
torch.testing.assert_close(
    conv_metrics["nvidia_2_4_sparsity/nvidia_eligible_block_fraction"],
    torch.tensor(1.0),
)

{
    "weight_shape": tuple(conv_model[0].weight.shape),
    "counts": count_row(conv_counts),
    "metrics": selected_scalars(
        conv_metrics,
        [
            "nvidia_2_4_sparsity/strict_block_fraction",
            "nvidia_2_4_sparsity/nvidia_eligible_block_fraction",
        ],
    ),
}

## Filtering And Layerwise Metrics

In [ ]:
class TinyFilteredModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bn = nn.BatchNorm1d(4)
        self.fc1 = nn.Linear(4, 2, bias=False)
        self.fc2 = nn.Linear(4, 1, bias=False)

    def forward(self, x):
        return self.fc1(self.bn(x)) + self.fc2(x)


filter_model = TinyFilteredModel()
with torch.no_grad():
    filter_model.fc1.weight.copy_(
        torch.tensor(
            [
                [0.0, 0.0, 1.0, 2.0],
                [3.0, 0.0, 4.0, 0.0],
            ]
        )
    )
    filter_model.fc2.weight.copy_(torch.tensor([[0.0, 0.0, 0.0, 1.0]]))

default_filter_metrics = track_2_4(filter_model)
layerwise_filter_metrics = track_2_4(filter_model, log_layerwise_stats=True)
included_fc1_metrics = track_2_4(
    filter_model,
    include=[filter_model.fc1],
    log_layerwise_stats=True,
)
ignored_fc2_metrics = track_2_4(
    filter_model,
    ignore=[filter_model.fc2],
    log_layerwise_stats=True,
)

assert all("/layers/" not in key for key in default_filter_metrics)
assert "nvidia_2_4_sparsity/layers/bn/strict_block_fraction" not in layerwise_filter_metrics
assert "nvidia_2_4_sparsity/layers/fc1/strict_block_fraction" in layerwise_filter_metrics
assert "nvidia_2_4_sparsity/layers/fc2/strict_block_fraction" in layerwise_filter_metrics
assert "nvidia_2_4_sparsity/layers/fc2/strict_block_fraction" not in included_fc1_metrics
assert "nvidia_2_4_sparsity/layers/fc2/strict_block_fraction" not in ignored_fc2_metrics
torch.testing.assert_close(
    default_filter_metrics["nvidia_2_4_sparsity/total_layers"],
    torch.tensor(2.0),
)
torch.testing.assert_close(
    layerwise_filter_metrics["nvidia_2_4_sparsity/layers/fc1/strict_block_fraction"],
    torch.tensor(1.0),
)
torch.testing.assert_close(
    layerwise_filter_metrics["nvidia_2_4_sparsity/layers/fc2/nvidia_eligible_block_fraction"],
    torch.tensor(1.0),
)
torch.testing.assert_close(
    included_fc1_metrics["nvidia_2_4_sparsity/strict_layers"],
    torch.tensor(1.0),
)
torch.testing.assert_close(
    ignored_fc2_metrics["nvidia_2_4_sparsity/strict_layers"],
    torch.tensor(1.0),
)

{
    "default_total_layers": scalar(default_filter_metrics["nvidia_2_4_sparsity/total_layers"]),
    "layerwise_keys": sorted(key for key in layerwise_filter_metrics if "/layers/" in key),
    "include_fc1": selected_scalars(
        included_fc1_metrics,
        [
            "nvidia_2_4_sparsity/strict_block_fraction",
            "nvidia_2_4_sparsity/strict_layers",
            "nvidia_2_4_sparsity/total_layers",
        ],
    ),
    "ignore_fc2": selected_scalars(
        ignored_fc2_metrics,
        [
            "nvidia_2_4_sparsity/strict_block_fraction",
            "nvidia_2_4_sparsity/strict_layers",
            "nvidia_2_4_sparsity/total_layers",
        ],
    ),
}

## Effective Parametrized Weights

In [ ]:
param_model = linear_with_weight(torch.tensor([[0.1, -0.2, 1.0, -1.0]]))
parametrize.register_parametrization(param_model[0], "weight", FakeQuantZeroSmall())

original_weight = param_model[0].parametrizations.weight.original.detach()
effective_weight = param_model[0].weight.detach()
param_counts = block_2_4_counts(param_model)
param_metrics = track_2_4(param_model)

assert int(original_weight.eq(0).sum()) == 0
assert int(effective_weight.eq(0).sum()) == 2
torch.testing.assert_close(param_counts, torch.tensor([[1.0, 1.0, 1.0, 0.0]]))
torch.testing.assert_close(
    param_metrics["nvidia_2_4_sparsity/strict_block_fraction"],
    torch.tensor(1.0),
)
torch.testing.assert_close(
    param_metrics["nvidia_2_4_sparsity/strict_layers"],
    torch.tensor(1.0),
)

{
    "original_weight": original_weight.tolist(),
    "effective_weight": effective_weight.tolist(),
    "counts": count_row(param_counts),
    "metrics": selected_scalars(
        param_metrics,
        [
            "nvidia_2_4_sparsity/strict_block_fraction",
            "nvidia_2_4_sparsity/strict_layers",
        ],
    ),
}

## MultiheadAttention Projections

In [ ]:
class TinyAttentionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=4, num_heads=2, batch_first=True)

    def forward(self, x):
        y, _ = self.attn(x, x, x, need_weights=False)
        return y


attention_model = TinyAttentionModel()
with torch.no_grad():
    attention_model.attn.in_proj_weight.fill_(1.0)
    attention_model.attn.in_proj_weight[:6, :] = 0.0

attention_metrics = track_2_4(
    attention_model,
    include=[attention_model.attn],
    ignore=[attention_model.attn.out_proj],
    log_layerwise_stats=True,
)

torch.testing.assert_close(
    attention_metrics["nvidia_2_4_sparsity/strict_block_fraction"],
    torch.tensor(0.0),
)
torch.testing.assert_close(
    attention_metrics["nvidia_2_4_sparsity/nvidia_eligible_block_fraction"],
    torch.tensor(0.5),
)
torch.testing.assert_close(
    attention_metrics["nvidia_2_4_sparsity/total_layers"],
    torch.tensor(1.0),
)
torch.testing.assert_close(
    attention_metrics["nvidia_2_4_sparsity/layers/attn/nvidia_eligible_blocks"],
    torch.tensor(6.0),
)
torch.testing.assert_close(
    attention_metrics["nvidia_2_4_sparsity/layers/attn/total_blocks"],
    torch.tensor(12.0),
)

{
    "in_proj_weight_shape": tuple(attention_model.attn.in_proj_weight.shape),
    "metrics": selected_scalars(
        attention_metrics,
        [
            "nvidia_2_4_sparsity/strict_block_fraction",
            "nvidia_2_4_sparsity/nvidia_eligible_block_fraction",
            "nvidia_2_4_sparsity/total_layers",
            "nvidia_2_4_sparsity/layers/attn/nvidia_eligible_blocks",
            "nvidia_2_4_sparsity/layers/attn/total_blocks",
        ],
    ),
}

## All Checks Passed

In [ ]:
all_checks_passed = True

{
    "all_checks_passed": all_checks_passed,
}